# API Rest
Los recursos web son muy útiles para compartir datos y ofrecer servicios. Por ejemplo, los usamos cuando escuchamos música en línea o hacemos cuentas con una calculadora web. Normalmente, las personas usan estos servicios desde una página web, con botones e imágenes, sin tener que saber nada de programación.

Pero a veces, necesitamos que una computadora haga cosas automáticamente, como buscar datos cada hora sin parar, o usar esos datos dentro de un sistema más grande. Abrir el navegador y hacerlo a mano no sería nada práctico. Ahí es donde entra algo llamado **API web**.

**API** significa *interfaz de programación de aplicaciones*. Es como un puente que permite que un programa (no una persona) se comunique con un servicio o datos en internet. Uno de los estilos más comunes de APIs se llama **REST**, que organiza todo en dos partes: el **cliente** (quien pide) y el **servidor** (quien responde).

El cliente solo necesita saber la dirección del recurso (eso se llama **URI**, o más comúnmente **URL**). Luego, hace una **solicitud HTTP**, que es como decirle al servidor: "¡Hola! ¿Me das esto, por favor?". Esa solicitud es simple, rápida y el servidor responde con la información.

Casi siempre, la respuesta viene en un formato llamado JSON, que es fácil de leer para personas y también para programas. Además, cada vez que se hace una solicitud, el servidor responde igual, sin importar quién la haya hecho antes. Todo funciona igual siempre.

Para pedir cosas o enviar datos, usamos métodos como **GET** (para obtener algo) y **POST** (para enviar o modificar algo).

Y para trabajar con estas APIs en Python, hay una librería muy práctica llamada `requests`. Con ella vamos a aprender a conectarnos y a obtener datos de internet fácilmente.

## Obtener datos con API
Digamos que necesitas información sobre los tipos de cambio actuales para calcular el valor de una cartera de inversiones en USD. ¿Cómo puedes obtener las tarifas de forma automática? En primer lugar, tenemos que encontrar una API web que proporcione datos en respuesta a las solicitudes correspondientes. Comenzaremos con una que no requiere ninguna autenticación (no requiere usuario y contraseña al conectarse).

Aquí está la URL base:  <https://api.frankfurter.app>. Parece un enlace de internet normal. Visita <https://frankfurter.app> para saber cómo usarla. Podemos descubrir que `/latest` nos dirige a las tarifas actuales. El uso básico es:

In [1]:
import requests
response = requests.get("https://api.frankfurter.app/latest")
print(response)

<Response [200]>


Aquí usamos la función `get()` de la librería `requests` incorporada para obtener algo del recurso proporcionado. Lo que recibimos fue un objeto `response`(documentado [aquí](https://requests.readthedocs.io/en/latest/api/#requests.Response)), que consta de varios conjuntos de información: código de estado, encabezado, contenido y otros. 

Cuando tratamos de examinar el objeto con `print()` vemos el **código de estado HTTP** por defecto. Este muestra cómo se ejecutó la solicitud y consta de tres dígitos:
* 200 significa que todo está bien,
* Mientras que 4XX y 5XX indican problemas del lado del cliente y del servidor.

Definitivamente, queremos acceder a más información, así que vamos a utilizar el método `json()` para analizar el contenido de la respuesta:

In [2]:
import requests
response = requests.get("https://api.frankfurter.app/latest")
print(response.json())

{'amount': 1.0, 'base': 'EUR', 'date': '2026-10-14', 'rates': {'AUD': 1.5971, 'BRL': 5.6246, 'CAD': 1.5918, 'CHF': 0.933, 'CNY': 7.4925, 'CZK': 24.357, 'DKK': 7.4754, 'GBP': 0.84385, 'HKD': 8.7859, 'HUF': 363.3, 'IDR': 20062.67, 'ILS': 3.4443, 'INR': 106.5283, 'ISK': 136.4, 'JPY': 177.39, 'KRW': 1496.54, 'MXN': 20.56, 'MYR': 4.5704, 'NOK': 10.6035, 'NZD': 1.9941, 'PHP': 70.393, 'PLN': 4.3658, 'RON': 5.3442, 'SEK': 11.158, 'SGD': 1.4359, 'THB': 37.587, 'TRY': 55.2921, 'USD': 1.1195, 'ZAR': 18.4382}}


Esto es mucho más útil para nuestro propósito: vemos el monto, la moneda base y la fecha, seguido de la cantidad de las otras monedas que se puede comprar por este monto. Vamos a reajustar la solicitud. ¿Cuántas libras se pueden comprar con $20? Al leer la documentación en el sitio [aquí](https://frankfurter.dev/es/), podemos aprender que debemos usar los parámetros `from` (desde), `to` (hasta) y `amount` (importe). Para mayor comodidad, `requests` permite pasar los parámetros de una petición HTTP como un diccionario Python normal:

In [3]:
import requests
# Definimos un diccionario de los parámetros con su clave-valor
parameters = {"from":"USD", "to":"GBP", "amount":20}

response = requests.get("https://api.frankfurter.app/latest", params=parameters)
print(response.json())


{'amount': 20.0, 'base': 'USD', 'date': '2026-10-14', 'rates': {'GBP': 15.0755}}


Además del código de estado y del contenido, las respuestas contienen `headers` (encabezados), que a su vez comprenden **metadatos** que, en este caso, son datos en protocolo HTTP. Ahí podemos encontrar información técnica sobre la respuesta: versión del servidor, tipo de contenido, programación, fecha, etc.

In [4]:
import requests
response = requests.get("https://api.frankfurter.app/latest")

print(f'Server: {response.headers['Server']}')
print(f'Content-Type: {response.headers['Content-Type']}')
print(f'Content-Encoding: {response.headers['Content-Encoding']}')

Server: cloudflare
Content-Type: application/json; charset=utf-8
Content-Encoding: gzip


Estas propiedades, especialmente el tipo de contenido, pueden ser útiles para comprender los detalles de una respuesta. Por ejemplo, un servidor web puede responder en un formato distinto de JSON (anotado en el encabezado como `application/json`), lo que nos serviría como pista (si no sabemos nada de la API) para tratar los datos de respuesta de forma diferente (por ejemplo, podría haber datos en texto plano o en formato binario).

## Autenticación
En los ejemplos anteriores se puede obtener datos sin ningún tipo de autenticación, es decir, el cliente no presenta ni verifica su identidad. Eso es bastante fuera de lo común porque muchos recursos web requieren algún tipo de autenticación para obtener datos de ellos o acceder a su funcionalidad más amplia, y muchas aplicaciones también pueden cobrar por su funcionalidad completa. Sin embargo, la mayoría de las APIs permiten obtener cierta información de forma gratuita.

Por lo general, la autenticación requiere la existencia de un secreto mutuo conocido solo por el cliente y el recurso web. Puede ser el clásico ejemplo de nombre de usuario y contraseña, o bien una **clave API** o un **token**.

### API Key
Es una clave fija que te da el servicio cuando te registras. Identifica a tu aplicación o proyecto.

Es como la credencial de un gimnasio: te la dan una vez al inscribirte, la muestras cada vez que entras y sirve hasta que la cancelas.

### Token

Es una clave temporal que obtienes al iniciar sesión (con usuario y contraseña, o con “Iniciar sesión con Google”). Normalmente representa a un usuario y caduca después de un tiempo, por ejemplo una hora.

Es como la pulsera de un concierto: te la ponen al entrar, te deja moverte sin volver a mostrar tu boleto y deja de servir cuando termina el evento.

Ambos se consideran secretos debido a la gran dificultad que supondría adivinar su valor exacto; hay demasiadas variantes como para intentar adivinarla aleatoriamente. Por eso las claves API y los tokens solo se comparten entre aquellas personas o sistemas que están estrictamente obligados a conocerla. Compartirla públicamente acabaría con su seguridad.

In [5]:
import requests

# Inicio sesión y recibo un token
login = requests.post('https://dummyjson.com/auth/login', json={'username': 'emilys', 'password': 'emilyspass'})

token = login.json()['accessToken']

# Usamos el token para pedir la información del usuario
headers = {'Authorization': f'Bearer {token}'}
me = requests.get('https://dummyjson.com/auth/me', headers=headers)

print(me.json()['firstName'])

Emily


### Ejercicio: obtener datos del Museo Metropolitano

La API web es una herramienta versátil utilizada por más que solo empresas: hay muchos museos, bibliotecas, proyectos científicos y entusiastas que ponen su información y servicios a disposición del público en general. Utiliza la librería `requests` y el endpoint proporcionado en la `url` para obtener información sobre una pintura de la colección del Museo Metropolitano. Examina la URL: el enlace base es `https://collectionapi.metmuseum.org/`, luego nos dirigimos a un recurso específico. En algún momento hay una parte `v1`. Esta es la versión de la API. Las diferentes versiones pueden actuar de manera diferente, por lo que es común reflejar la actual. Finalmente, el número `437133` es la identificación de la pintura.

1. Importa `requests`.
2. Llama al endpoint mediante la `url` proporcionada. Recupera el resultado en formato JSON aplicándole el método apropiado. Guarda la JSON resultante en la variable `response`.
3. Muestra `artistDisplayName` desde JSON llamándolo por la clave.

In [6]:
import requests
base_url = 'https://collectionapi.metmuseum.org/'
url = base_url + 'public/collection/v1/objects/437133'

response = requests.get(url)
print(response.json()['artistDisplayName'])

Claude Monet


### Ejercicio: departamentos del Museo Metropolitano

Para llegar al endpoint `public/collection/v1/departments` utilizando la misma URL base, simplemente agrega `public/collection/v1/departments` a la URL base. Guarda la URL resultante en la variable `url`.

A continuación, en el precódigo, verás una línea donde almacenamos la respuesta del endpoint y la guardamos en la variable `response`.

Tu objetivo ahora es convertir la respuesta en un archivo JSON. Luego, itera sobre `response.json()['departments']` y muestra los que tienen `'Art'` en `dpt['displayName']`.

In [7]:
import requests

url = 'https://dummyjson.com/products'

params  = {'limit':3}
response = requests.get(url, params)
print(response.json())

{'products': [{'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Lucas Gordon', 'reviewerEmail': 'lucas.gordon@x.dummyjson.com'}, {'rating': 5